# Object Tracking with Your Webcam (and How to Judge the AI)

In this notebook your laptop camera will **find objects, give each one a number (ID), and follow them** as they move.
Then we will ask an important question: **how good is this AI, really?**

**Part 1: Live demo**
- a **box** around every object, its **class** (person, cup, phone...) and its **confidence**
- a unique **ID** that stays the same while the object moves
- a colored **trajectory** (the path the object walked)
- live information: **frame number**, **FPS**, **speed**, and **how many objects** are visible

**Part 2: Evaluation**
- which numbers can we show **live**, and which ones we must not
- the main metrics explained with simple stories
- a **comparison of two models**: `yolo11n` (nano) against `yolo11s` (small)
- a **live side-by-side demo** of both models on the same camera

## What is object tracking?

Imagine you look at a street through a window and you give a name to every car: "car 1", "car 2"...
Even when the cars move, you still know that car 1 is car 1. This is exactly what tracking does.

| Task | Question it answers | Example |
|---|---|---|
| **Detection** | "What is in this picture, and where?" | "There is a person here" |
| **Tracking** | "Is this the same object as in the previous picture?" | "This is still person number 3" |

Detection works on **one picture**. Tracking works on a **video** (many pictures in a row) and connects them.

## Step 0: Prepare your computer (one time only)

Open a terminal in the folder of this notebook and create a **virtual environment**.
It is a clean box where we install only what this project needs, so nothing breaks on your computer.

```bash
# 1. Create the environment
python -m venv .venv

# 2. Activate it
#    Linux / macOS:
source .venv/bin/activate
#    Windows (PowerShell):
.venv\Scripts\Activate.ps1

# 3. Install the tools
pip install ipykernel ultralytics opencv-python
```

Then in **VS Code**: open this notebook, click **Select Kernel** (top right), choose **Python Environments**, and pick **.venv**.

**Camera permission** (only if the camera does not open):

- **Windows**: Settings > Privacy & security > Camera > allow desktop apps
- **macOS**: System Settings > Privacy & Security > Camera > enable **Visual Studio Code** (or Terminal), then restart VS Code
- **Linux**: usually works directly

The cell below does the install for you if you skipped step 3. It can take a few minutes the first time.

In [ ]:
%pip install -q ultralytics opencv-python

## Step 1: Import the tools

Each library has one job:

- `cv2` (OpenCV): talk to the camera and draw on images
- `ultralytics`: the AI models that detect and track objects (YOLO)
- `numpy`: fast maths on numbers and images
- `matplotlib`: draw the charts of Part 2
- `time`, `collections`: small helpers for the clock, counting, and remembering past positions

In [ ]:
import time
from collections import defaultdict, deque, Counter
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display, clear_output, Image as NotebookImage

print("OpenCV version:", cv2.__version__)
print("All tools loaded.")

## Step 2: Settings (the only cell you need to change)

Think of this as the **control panel**. Change a value, run the cell, and see what happens.

| Setting | What it does |
|---|---|
| `MODEL_NAME` | The AI brain used for tracking. `yolo11n.pt` is the smallest and fastest |
| `COMPARE_MODEL_NAME` | The second brain, used in Part 2 to compare. `yolo11s.pt` is bigger and more accurate |
| `CAMERA_INDEX` | `0` is the built-in camera. Try `1` for an external USB camera |
| `CONFIDENCE` | Minimum sureness (0 to 1). Higher = fewer but safer detections |
| `IMAGE_SIZE` | Size the model looks at. Smaller = faster, bigger = more accurate |
| `TRAIL_LENGTH` | How many past positions to draw as the trajectory |
| `CLASSES_TO_TRACK` | `None` tracks everything. Or write a list like `["person", "cup"]` |
| `MIRROR` | `True` flips the image like a mirror, which feels natural |
| `DISPLAY_MODE` | `"window"` opens a separate window (fast, press **Q** to stop). `"notebook"` shows video inside the notebook |
| `RUN_REAL_EVALUATION` | `True` runs a real accuracy test in Part 2 (needs internet and about 1 to 3 minutes) |

In [ ]:
MODEL_NAME          = "yolo11n.pt"   # nano model: small and fast (downloads automatically the first time)
COMPARE_MODEL_NAME  = "yolo11s.pt"   # small model: bigger, slower, more accurate
CAMERA_INDEX        = 0              # 0 = built-in laptop camera
CONFIDENCE          = 0.4            # ignore detections below 40% sureness
IMAGE_SIZE          = 640            # smaller = faster, bigger = more accurate
TRAIL_LENGTH        = 40             # how many past positions we draw
CLASSES_TO_TRACK    = None           # None = everything, or for example ["person", "cell phone", "cup"]
MIRROR              = True           # flip the camera like a mirror
DISPLAY_MODE        = "window"       # "window" or "notebook"
MAX_SECONDS         = None           # None = run until you stop it, or put 60 for one minute
RUN_REAL_EVALUATION = False          # True = measure real accuracy in Part 2 (slower)

## Step 3: Load the AI models

We use **YOLO** ("You Only Look Once"). It looks at a full image one time and instantly tells us
all the objects inside it. It was trained on the **COCO** dataset, which contains **80 everyday classes**
(person, bottle, cup, phone, laptop, chair, dog, car...).

We load **two models** from the same family. They work exactly the same way, only their size is different:

- **yolo11n** (nano): the light one, used for the live tracking
- **yolo11s** (small): the bigger one, used for the comparison in Part 2

The first time, the files are downloaded (6 MB and 19 MB). After that everything works **offline**.

In [ ]:
model = YOLO(MODEL_NAME)                       # this one is used for tracking

compare_models = {                             # these two are used in Part 2
    Path(MODEL_NAME).stem: model,
    Path(COMPARE_MODEL_NAME).stem: YOLO(COMPARE_MODEL_NAME),
}

# Convert class names (like "person") to the numbers the model uses internally
if CLASSES_TO_TRACK:
    class_ids = [i for i, name in model.names.items() if name in CLASSES_TO_TRACK]
else:
    class_ids = None


def count_params(m):
    """Parameters = tiny 'knobs' inside the model that were adjusted during training."""
    return sum(p.numel() for p in m.model.parameters())


print(f"The model knows {len(model.names)} classes. Examples:", list(model.names.values())[:10])
for label, m in compare_models.items():
    print(f"{label}: {count_params(m) / 1e6:.1f} million parameters")

## Step 4: Test the camera

Before the real show, let's take **one photo** to check that the camera works.
If you see your picture, everything is fine. If you see an error, read the permission tips in Step 0.

In [ ]:
def open_camera():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        raise RuntimeError(
            "Cannot open the camera. Check: (1) camera permission for VS Code, "
            "(2) no other app is using the camera, (3) try CAMERA_INDEX = 1."
        )
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    return cap


def take_photo():
    """Take one picture with the camera (the first frames are dark, so we skip them)."""
    cap = open_camera()
    time.sleep(0.5)
    ok = False
    for _ in range(5):
        ok, photo = cap.read()
    cap.release()
    if not ok:
        return None
    return cv2.flip(photo, 1) if MIRROR else photo


photo = take_photo()
if photo is not None:
    print("Camera works. Picture size:", photo.shape[1], "x", photo.shape[0], "pixels")
    display(NotebookImage(data=cv2.imencode(".jpg", photo)[1].tobytes(), width=500))
else:
    print("The camera opened but gave no picture. Try again or change CAMERA_INDEX.")

## Step 5: How does the tracker remember objects?

Every time the camera gives us a **frame** (one picture of the video), three things happen:

1. **Detect**: YOLO finds all the boxes in the new frame.
2. **Match**: The tracker compares the new boxes with the boxes from the previous frame.
   A box that is almost in the same place is probably the **same object**.
3. **Label**: Matched boxes keep their old ID. New objects get a **new ID**.

The tracker used here is called **ByteTrack**. It is fast, and it even keeps low-confidence boxes for a moment,
which helps to not lose an object when it is partly hidden.

```
Frame 1:  [person] -> ID 1
Frame 2:  [person] (moved a little) -> still ID 1
Frame 3:  [person] [cup] -> ID 1 stays, cup gets ID 2
```

**Important**: the tracker follows *position and movement*, not the face or the identity.
If an object leaves the screen and comes back later, it can receive a new ID.

## Step 6: Drawing helpers

These small functions only **draw** on the image. They do not change the AI.

- `color_for_id`: every ID gets its own color, so you can follow it with your eyes
- `draw_box_and_label`: the box around the object and a tag like `ID 3 | person 91%`
- `draw_trail`: the path of the object (older parts are thinner, recent parts are thicker)
- `draw_info_panel`: the dark panel with frame number, FPS, speed, confidence, and counts
- `show_frame` and `close_camera`: show a picture, and give the camera back when we finish

In [ ]:
FONT = cv2.FONT_HERSHEY_SIMPLEX

def color_for_id(track_id):
    """Same ID always gives the same bright color."""
    rng = np.random.RandomState(int(track_id) * 7 + 3)
    return tuple(int(c) for c in rng.randint(80, 255, 3))


def draw_box_and_label(frame, box, track_id, name, conf, color):
    x1, y1, x2, y2 = box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    text = f"ID {track_id} | {name} {conf:.0%}"
    (w, h), _ = cv2.getTextSize(text, FONT, 0.55, 1)
    top = max(y1 - h - 10, 0)
    cv2.rectangle(frame, (x1, top), (x1 + w + 8, top + h + 10), color, -1)
    cv2.putText(frame, text, (x1 + 4, top + h + 3), FONT, 0.55, (0, 0, 0), 1, cv2.LINE_AA)


def draw_trail(frame, points, color):
    points = list(points)
    for i in range(1, len(points)):
        thickness = max(1, int(1 + 4 * i / len(points)))   # newer = thicker
        cv2.line(frame, points[i - 1], points[i], color, thickness, cv2.LINE_AA)


def draw_info_panel(frame, frame_number, fps, latency_ms, avg_conf, class_counts, total):
    lines = [
        f"Frame: {frame_number}",
        f"FPS: {fps:.1f}",
        f"Speed: {latency_ms:.0f} ms per frame",
        f"Avg confidence: {avg_conf:.0%}",
        f"Objects on screen: {total}",
    ]
    lines += [f"   {name}: {count}" for name, count in class_counts.items()]
    lines.append("Press Q to quit" if DISPLAY_MODE == "window" else "Stop: interrupt the cell")

    panel_height = 16 + 24 * len(lines)
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (300, 10 + panel_height), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)      # see-through dark panel
    for i, text in enumerate(lines):
        cv2.putText(frame, text, (20, 36 + 24 * i), FONT, 0.6, (255, 255, 255), 1, cv2.LINE_AA)


def show_frame(frame, title):
    """Show one picture. Returns True if the user asked to stop."""
    if DISPLAY_MODE == "window":
        cv2.imshow(title, frame)
        return cv2.waitKey(1) & 0xFF in (ord("q"), ord("Q"), 27)   # Q or Esc
    clear_output(wait=True)
    display(NotebookImage(data=cv2.imencode(".jpg", frame)[1].tobytes()))
    return False


def close_camera(cap):
    cap.release()                           # always give the camera back
    cv2.destroyAllWindows()
    for _ in range(4):
        cv2.waitKey(1)                      # lets the window close properly on macOS

print("Helpers ready.")

## Step 7: The main tracking loop

This is the heart of the notebook. It repeats the same steps again and again, many times per second:

1. **Read** a frame from the camera
2. **Track**: `model.track(...)` detects objects and gives them IDs
3. **Remember** the center of each object to build its trajectory
4. **Draw** boxes, labels, trails, and the info panel
5. **Show** the result
6. **Stop** when you press **Q** (or after `MAX_SECONDS`)

At the end it prints a small **report** of everything that was seen during the session.

In [ ]:
def run_tracker():
    cap = open_camera()

    trails = defaultdict(lambda: deque(maxlen=TRAIL_LENGTH))   # ID -> last positions
    class_of_id = {}                                           # ID -> class name
    frames_alive = Counter()                                   # ID -> how many frames it lived
    frame_number = 0
    fps = 0.0
    start = last_time = time.time()

    try:
        while True:
            # 1. READ a frame from the camera
            ok, frame = cap.read()
            if not ok:
                print("No more pictures from the camera.")
                break
            if MIRROR:
                frame = cv2.flip(frame, 1)
            frame_number += 1

            # 2. TRACK: detect objects and give them IDs
            #    persist=True means "remember the previous frames"
            result = model.track(
                frame, persist=True, tracker="bytetrack.yaml",
                conf=CONFIDENCE, imgsz=IMAGE_SIZE, classes=class_ids, verbose=False
            )[0]
            latency_ms = sum(result.speed.values())     # time the AI needed for this frame

            class_counts = Counter()
            confs = np.array([])
            boxes = result.boxes

            if boxes is not None and boxes.id is not None:
                xyxy    = boxes.xyxy.cpu().numpy().astype(int)    # box corners
                ids     = boxes.id.cpu().numpy().astype(int)      # tracking IDs
                classes = boxes.cls.cpu().numpy().astype(int)     # class numbers
                confs   = boxes.conf.cpu().numpy()                # sureness

                for box, track_id, cls, conf in zip(xyxy, ids, classes, confs):
                    name = model.names[cls]
                    color = color_for_id(track_id)
                    class_of_id[track_id] = name
                    frames_alive[track_id] += 1
                    class_counts[name] += 1

                    # 3. REMEMBER the center of the box
                    center = (int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2))
                    trails[track_id].append(center)

                    # 4. DRAW
                    draw_trail(frame, trails[track_id], color)
                    draw_box_and_label(frame, box, track_id, name, conf, color)
                    cv2.circle(frame, center, 4, color, -1)

            # FPS = how many frames we process per second (smoothed so it does not jump)
            now = time.time()
            current_fps = 1.0 / max(now - last_time, 1e-6)
            fps = current_fps if fps == 0 else 0.9 * fps + 0.1 * current_fps
            last_time = now

            avg_conf = float(confs.mean()) if len(confs) else 0.0
            draw_info_panel(frame, frame_number, fps, latency_ms, avg_conf,
                            class_counts, sum(class_counts.values()))

            # 5. SHOW the picture
            if show_frame(frame, "Object Tracking  (press Q to quit)"):
                break

            # 6. STOP after the time limit (if one was set)
            if MAX_SECONDS and now - start > MAX_SECONDS:
                break

    except KeyboardInterrupt:
        print("Stopped by you.")

    finally:
        close_camera(cap)

    # Session report
    duration = time.time() - start
    lives = list(frames_alive.values())
    print("\n----- Session report -----")
    print(f"Duration        : {duration:.1f} seconds")
    print(f"Frames processed: {frame_number}")
    print(f"Average FPS     : {frame_number / max(duration, 1e-6):.1f}")
    print(f"Different objects tracked: {len(class_of_id)}")
    for name, count in Counter(class_of_id.values()).items():
        print(f"   {name}: {count}")
    if lives:
        short = sum(1 for n in lives if n < 15)
        print(f"Average life of an ID: {np.mean(lives):.0f} frames")
        print(f"Short-lived IDs (under 15 frames): {short}   (many of them = a shaky tracker)")

print("Tracker ready. Run the next cell to start.")

## Step 8: Run the live demo

Run the cell below, then try these:

- Stand in front of the camera and **walk left and right**: watch your trajectory.
- Hold a **phone, cup, bottle, or book** in your hand and move it.
- Bring **a friend** into the picture: each person gets a different ID and color.
- **Hide** an object behind something for a second and see if the ID survives.
- Look at the panel: **FPS**, **speed**, and **average confidence** change live.

Press **Q** in the video window to stop.
(If you chose `DISPLAY_MODE = "notebook"`, stop with the square button at the top of the notebook.)

In [ ]:
run_tracker()

# Part 2: How good is this AI?

## Step 9: Should we show evaluation metrics on the live camera?

**Short answer: show only the live-friendly numbers. Never show accuracy numbers live.**

To give a grade, a teacher needs the **answer sheet**. For accuracy metrics, the answer sheet is a set of images where
humans already drew the correct boxes (we call it the **ground truth**). Your webcam has no answer sheet.
If the screen said "Accuracy: 92%" it would be **invented**.

| Number | Needs an answer sheet? | Show it live? |
|---|---|---|
| FPS and speed (ms) | No | **Yes** |
| Number of objects | No | **Yes** |
| Confidence | No, it is the model's own feeling | **Yes**, but say it is *sureness*, not *correctness* |
| ID stability (session report) | No | At the end |
| Precision, recall, mAP | **Yes** | **No**. Measure them on a test set (Steps 12 and 13) |

**The confidence trap**: a model can be 95% sure and still be wrong, like a student who is very confident
and writes the wrong answer. That is why confidence is not accuracy.

## Step 10: The metrics as a story

Imagine a **security guard** who watches a camera screen. His only job is to shout **"PERSON!"** every time he sees one.

| Metric | The question | Guard example | When it is low |
|---|---|---|---|
| **Precision** | When he shouts, how often is he right? | He shouts 8 times, 6 are real people: **75%** | Many **false alarms** |
| **Recall** | Of all the real people, how many did he notice? | 10 people are there, he noticed 6: **60%** | He **misses** people |
| **IoU** | How well does the box fit the object? | Two sheets of paper: the overlap divided by the total covered area | The box is **shifted or too big** |
| **mAP** | One overall grade for everything above | The **school report card** out of 100 | The model is weak overall |
| **FPS** | How many pictures can it look at per second? | How fast the guard's eyes move | Video is **laggy** |

**mAP50** is an easy exam: a box counts as correct if it covers at least half of the object.
**mAP50-95** is the strict exam: the same test repeated with stricter and stricter fitting (from 50% to 95%), then averaged.
This is the main number used in the official model tables.

**Precision and recall pull in opposite directions.** A guard who shouts at everything finds all people (high recall)
but makes many false alarms (low precision). A guard who shouts only when 100% sure makes no false alarms
but misses people. Our `CONFIDENCE` setting is exactly this dial.

**Tracking has its own scorecard**, but it needs labelled videos:

| Name | Plain meaning |
|---|---|
| ID switch | The same object suddenly received a new number |
| IDF1 | How well IDs stay on the right object |
| MOTA | One overall grade for tracking |

In this notebook we use a simple stand-in: the **average life of an ID** in the session report.

Run the next cell to **see** IoU and to compute precision and recall yourself.

In [ ]:
def iou(a, b):
    """IoU = overlap area / total covered area. Boxes are (x1, y1, x2, y2)."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    overlap = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return overlap / (area_a + area_b - overlap)


truth = (20, 20, 120, 120)    # the box a human drew (correct answer)
guesses = [("Bad fit", (80, 70, 180, 170)),
           ("OK fit", (40, 35, 140, 135)),
           ("Great fit", (24, 22, 124, 122))]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (title, guess) in zip(axes, guesses):
    for box, color, name in [(truth, "green", "Human (correct)"), (guess, "orange", "Model guess")]:
        ax.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                   fill=False, edgecolor=color, linewidth=3, label=name))
    ax.set_xlim(0, 200)
    ax.set_ylim(200, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"{title}\nIoU = {iou(truth, guess):.2f}", fontsize=12)
axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

# Precision and recall with the guard example
real_people = 10     # people really in the picture
shouts      = 8      # times the model said "person!"
correct     = 6      # shouts that were right

print(f"Precision = correct / shouts       = {correct} / {shouts}       = {correct / shouts:.0%}")
print(f"Recall    = correct / real people  = {correct} / {real_people}      = {correct / real_people:.0%}")

## Step 11: Two models, same camera, same picture

`yolo11n` and `yolo11s` are from the same family and use the same code. The difference is the **size of the brain**:

- **yolo11n (nano)**: like a **pocket notebook**. Very quick to open, but it contains fewer details.
- **yolo11s (small)**: like a **full textbook**. More details and more careful, but slower to read.

The size is counted in **parameters**: the tiny knobs inside the model that were adjusted during training.
More parameters = bigger brain = more memory and more time.

The next cell takes **one photo** with your camera (put 2 or 3 objects in front of it first), gives the **same photo to both models**,
and measures the speed on **your** computer. If the camera is not available, it uses a sample photo of a bus.

In [ ]:
# 1. Get one test picture
try:
    test_image = take_photo()
except RuntimeError:
    test_image = None
if test_image is None:
    from ultralytics.utils import ASSETS
    test_image = cv2.imread(str(ASSETS / "bus.jpg"))
    print("Camera not available: using the sample bus photo.")

# 2. Give the same picture to both models and measure
RUNS = 15
results_table = {}
annotated = {}

for label, m in compare_models.items():
    m.predict(test_image, imgsz=IMAGE_SIZE, verbose=False)            # warm-up (first run is always slower)
    times = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        res = m.predict(test_image, imgsz=IMAGE_SIZE, conf=CONFIDENCE, classes=class_ids, verbose=False)[0]
        times.append((time.perf_counter() - t0) * 1000)
    confs = res.boxes.conf.cpu().numpy()
    results_table[label] = {
        "ms": float(np.mean(times)),
        "fps": 1000 / float(np.mean(times)),
        "objects": len(confs),
        "avg_conf": float(confs.mean()) if len(confs) else 0.0,
        "params_m": count_params(m) / 1e6,
    }
    annotated[label] = res.plot()                                      # the picture with boxes drawn

# 3. Print the numbers
print(f"{'Model':<10}{'Time per picture':>18}{'FPS':>8}{'Objects found':>16}{'Avg confidence':>16}")
for label, r in results_table.items():
    print(f"{label:<10}{r['ms']:>15.0f} ms{r['fps']:>8.1f}{r['objects']:>16}{r['avg_conf']:>16.0%}")

# 4. Show both results side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (label, img) in zip(axes, annotated.items()):
    r = results_table[label]
    ax.imshow(img[:, :, ::-1])                                         # BGR -> RGB for matplotlib
    ax.axis("off")
    ax.set_title(f"{label}: {r['objects']} objects found in {r['ms']:.0f} ms", fontsize=13)
plt.tight_layout()
plt.show()

### The comparison in three charts

Three simple questions, one chart each:

1. **Speed**: how many pictures per second? (measured on **your** laptop just now)
2. **Size**: how big is the brain? (measured from the loaded models)
3. **Score**: what is the report card? This is the official **mAP50-95** on the COCO test set, published by Ultralytics
   (we cannot measure it with a webcam, see Step 9).

In [ ]:
# Official scores on the COCO validation set (5,000 images the models never studied). Source: Ultralytics docs.
OFFICIAL_MAP = {"yolo11n": 39.5, "yolo11s": 47.0, "yolo11m": 51.5}

labels = list(results_table)
colors = ["#4C78A8", "#F58518"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

charts = [
    ("SPEED\nPictures per second (higher is better)", [results_table[l]["fps"] for l in labels], "%.0f"),
    ("SIZE\nMillions of knobs in the brain (smaller = lighter)", [results_table[l]["params_m"] for l in labels], "%.1f"),
    ("SCORE\nReport card out of 100, mAP50-95 (higher is better)", [OFFICIAL_MAP.get(l, 0) for l in labels], "%.1f"),
]
for ax, (title, values, fmt) in zip(axes, charts):
    bars = ax.bar(labels, values, color=colors)
    ax.bar_label(bars, fmt=fmt, fontsize=13, fontweight="bold")
    ax.set_title(title, fontsize=11)
    ax.set_yticks([])
    ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

# The verdict in plain words
fast, slow = sorted(labels, key=lambda l: results_table[l]["fps"], reverse=True)
big, small = sorted(labels, key=lambda l: results_table[l]["params_m"], reverse=True)
print(f"Speed : {fast} is {results_table[fast]['fps'] / results_table[slow]['fps']:.1f}x faster than {slow} on this computer.")
print(f"Size  : {big} is {results_table[big]['params_m'] / results_table[small]['params_m']:.1f}x bigger than {small}.")
if all(l in OFFICIAL_MAP for l in labels):
    best, other = sorted(labels, key=lambda l: OFFICIAL_MAP[l], reverse=True)
    print(f"Score : {best} gets {OFFICIAL_MAP[best] - OFFICIAL_MAP[other]:.1f} points more than {other} on the official test.")
print("\nThe lesson: a bigger brain is usually more accurate but slower. There is no perfect model, only the right one for the job.")

## Step 12 (optional): Measure the accuracy yourself

Now we do what a researcher does: use an **answer sheet**.
`coco128` is a small collection of **128 images** with the correct boxes already drawn by humans.
Both models take the exam and we measure **precision, recall, mAP50, and mAP50-95**.

Set `RUN_REAL_EVALUATION = True` in Step 2, run Step 2 again, then run the cell below.
It downloads `coco128` (about 7 MB) the first time.

**An honest warning**: these 128 images come from the data the models used for training, so the scores will look
**better than real life**, like a student who practiced on the same questions.
The official numbers in Step 11 use images that the models have never seen.

In [ ]:
if not RUN_REAL_EVALUATION:
    print("Skipped. Set RUN_REAL_EVALUATION = True in Step 2 to run the real exam.")
else:
    scores = {}
    try:
        for file in (MODEL_NAME, COMPARE_MODEL_NAME):
            label = Path(file).stem
            print(f"{label} is taking the exam...")
            metrics = YOLO(file).val(data="coco128.yaml", imgsz=IMAGE_SIZE, batch=16,
                                     workers=0, plots=False, verbose=False)
            scores[label] = {
                "Precision": metrics.box.mp,       # average over all classes
                "Recall": metrics.box.mr,
                "mAP50": metrics.box.map50,
                "mAP50-95": metrics.box.map,
            }
    except Exception as error:
        print("The exam could not run (it needs internet to download coco128):", error)

    if scores:
        names = list(next(iter(scores.values())))
        x = np.arange(len(names))
        fig, ax = plt.subplots(figsize=(9, 4))
        for i, (label, values) in enumerate(scores.items()):
            bars = ax.bar(x + (i - 0.5) * 0.38, [values[n] * 100 for n in names], 0.38,
                          label=label, color=["#4C78A8", "#F58518"][i])
            ax.bar_label(bars, fmt="%.0f", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(names, fontsize=12)
        ax.set_yticks([])
        ax.set_title("Real exam on 128 images (out of 100, higher is better)")
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.legend()
        plt.tight_layout()
        plt.show()

## Step 13: Live side-by-side demo

The best way to **feel** the difference: both models look at **the same live frame**, at the same time.

Watch the banner on each side: **FPS**, **number of objects found**, and **average confidence**.
Try hard cases: small objects far from the camera, objects partly hidden, or a messy desk.

Press **Q** to stop. (Because two models work on every frame, the FPS will be lower than in Step 8. This is normal.)

In [ ]:
def run_live_comparison():
    cap = open_camera()
    start = time.time()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if MIRROR:
                frame = cv2.flip(frame, 1)

            panels = []
            for label, m in compare_models.items():
                res = m.predict(frame, imgsz=IMAGE_SIZE, conf=CONFIDENCE, classes=class_ids, verbose=False)[0]
                ms = sum(res.speed.values())
                confs = res.boxes.conf.cpu().numpy()
                avg = float(confs.mean()) if len(confs) else 0.0

                panel = res.plot()                                         # picture with boxes
                height = int(panel.shape[0] * 640 / panel.shape[1])
                panel = cv2.resize(panel, (640, height))
                cv2.rectangle(panel, (0, 0), (640, 34), (0, 0, 0), -1)
                banner = f"{label} | {1000 / ms:.0f} FPS | {len(confs)} objects | confidence {avg:.0%}"
                cv2.putText(panel, banner, (10, 24), FONT, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
                panels.append(panel)

            if show_frame(np.hstack(panels), "Model comparison  (press Q to quit)"):
                break
            if MAX_SECONDS and time.time() - start > MAX_SECONDS:
                break
    except KeyboardInterrupt:
        print("Stopped by you.")
    finally:
        close_camera(cap)


run_live_comparison()

## Step 14: Three numbers to remember

If the audience remembers only three things, they should be these:

| | Kid words | Normal words | Pro words |
|---|---|---|---|
| **Speed** | How fast does it look? | FPS (pictures per second) | Latency (ms) and FPS |
| **Guess** | How sure is it? | Confidence | Confidence score (not accuracy) |
| **Score** | How good is its report card? | Accuracy | mAP50-95, precision, recall |

**Which model should I choose?**

| If you need... | Choose |
|---|---|
| Smooth live video on a normal laptop, phone, or small device | **yolo11n** (nano) |
| Better detection of small or difficult objects, and you have a stronger computer | **yolo11s** (small) |
| The best accuracy, and speed is not a problem | A bigger model (`yolo11m`, `yolo11l`, `yolo11x`) |

## Step 15: Recap and troubleshooting

| On the screen | Meaning |
|---|---|
| Colored **box** | Where the model thinks the object is |
| **ID 3** | The tracker's number for that object. It stays while the object stays visible |
| **person 91%** | The class, and how sure the model is (confidence) |
| Colored **tail** | The trajectory: where the object was during the last moments |
| **Frame** | How many pictures we have processed since the start |
| **FPS** | Frames per second. Around 15 to 30 feels smooth |
| **Speed** | Milliseconds the AI needs for one frame (lower is better) |
| **Avg confidence** | The average sureness of all objects on screen |

### Things to try

- Change `CLASSES_TO_TRACK` to `["person"]` to follow only people.
- Raise `CONFIDENCE` to `0.7`: fewer boxes (higher precision), but more objects missed (lower recall).
- Lengthen `TRAIL_LENGTH` to `120` for long, beautiful trajectories.
- Set `COMPARE_MODEL_NAME = "yolo11m.pt"` and rerun Part 2 to see an even bigger model.

### Common problems

| Problem | Solution |
|---|---|
| "Cannot open the camera" | Allow camera access for VS Code, close Zoom/Teams/Meet, try `CAMERA_INDEX = 1` |
| Video is slow (low FPS) | Use `yolo11n.pt`, lower `IMAGE_SIZE` to `320`, or improve the room light |
| The window does not appear (macOS) | Use `DISPLAY_MODE = "notebook"` |
| The ID changes for the same object | Normal when the object is hidden for a long time or leaves the screen |
| Wrong class (cup seen as bottle) | Normal for a small model. Use a bigger model or raise `CONFIDENCE` |
| Step 12 fails | It needs internet to download `coco128`. Skip it, it is optional |

### Limits to remember

The models only know the **80 COCO classes**. They cannot recognize a "pen" well, and they
track *positions*, not *who* someone is. In the next notebooks we will see other computer vision tasks.